# faster-whisper + MFCC/KMeans로 화자별 회의록 만들기

이 노트북은 `S00007844.wav` 파일을 기준으로 다음 두 작업을 한 번에 실습합니다.

1. **faster-whisper**로 음성을 텍스트로 변환합니다.
2. **MFCC + KMeans**로 목소리 특징을 군집화해서 화자분리 표를 만듭니다.
3. Whisper 전사 결과와 MFCC/KMeans 화자분리 결과를 시간 겹침 기준으로 결합합니다.

## 최종 출력 파일

- `S00007844_faster_whisper_transcript.csv`  
  Whisper 전사 결과입니다.
- `S00007844_mfcc_kmeans_segments.csv`  
  MFCC/KMeans로 만든 원본 화자분리 구간 표입니다.
- `S00007844_mfcc_kmeans_segments_merged.csv`  
  같은 화자가 이어지는 구간을 병합한 화자분리 표입니다.
- `S00007844_speaker_transcript.csv`  
  최종 화자별 전사 결과입니다.

## 핵심 개념

- **Whisper / STT**: 사람이 말한 내용을 텍스트로 바꾸는 기술입니다.
- **MFCC**: 목소리의 주파수 특징을 숫자로 뽑는 방법입니다.
- **KMeans**: 비슷한 MFCC 특징끼리 묶어서 Speaker_00, Speaker_01처럼 군집을 만드는 방법입니다.
- **시간 겹침 결합**: Whisper 문장 시간과 화자 구간 시간을 비교해서 가장 많이 겹치는 화자를 붙이는 방법입니다.

주의: MFCC + KMeans 화자분리는 교육용 기초 방식입니다. 실제 서비스 품질은 pyannote.audio, WhisperX, NVIDIA NeMo 같은 전용 화자분리 모델이 더 좋습니다.

## 0. 준비 사항

처음 실행하는 환경에서는 아래 셀에서 필요한 라이브러리를 설치합니다.

> GPU가 있으면 더 빠르게 실행됩니다. CPU에서도 실행 가능하지만 시간이 더 걸릴 수 있습니다.


In [1]:
# ============================================
# 0. 필요한 패키지 설치
# ============================================
# 이 셀은 노트북 실행에 필요한 라이브러리를 설치합니다.
#
# faster-whisper : Whisper 계열 음성인식(STT)을 빠르게 실행하기 위한 라이브러리
# pandas         : 표 형태의 데이터를 다루기 위한 라이브러리
# tqdm           : 진행률 표시용 라이브러리
# librosa        : 오디오 로드, MFCC 추출 등 음성 특징 추출용 라이브러리
# soundfile      : 오디오 파일 읽기/쓰기 보조 라이브러리
# scikit-learn   : KMeans, StandardScaler 같은 머신러닝 도구 제공
# numpy          : 숫자 배열 계산용 라이브러리
#
# 이미 설치되어 있다면 이 셀을 다시 실행해도 큰 문제는 없습니다.
# 다만 수업 환경에서는 설치 시간이 걸릴 수 있으므로, 한 번 설치 후에는 생략해도 됩니다.

import sys  # 현재 노트북이 사용하는 파이썬 실행 파일 경로를 알기 위해 사용합니다.

# ! 명령은 Jupyter Notebook에서 터미널 명령을 실행할 때 사용합니다.
# {sys.executable}을 쓰면 현재 노트북 커널과 같은 파이썬 환경에 패키지를 설치할 수 있습니다.
!{sys.executable} -m pip install -U faster-whisper pandas tqdm librosa soundfile scikit-learn numpy


## 1. 음성 파일 확인

현재 노트북은 `/mnt/data/S00007844.wav` 파일을 기준으로 작성되었습니다.

로컬 PC에서 실행하는 경우에는 `AUDIO_PATH`를 본인의 파일 경로로 바꾸면 됩니다.


In [4]:
# ============================================
# 1. 음성 파일 경로 확인 및 기본 정보 출력
# ============================================
# 이 셀의 목적은 다음과 같습니다.
# 1) S00007844.wav 파일이 어디에 있는지 찾기
# 2) 파일이 실제로 존재하는지 확인하기
# 3) wav 파일의 기본 정보, 예: 채널 수, 샘플링레이트, 길이를 출력하기

from pathlib import Path  # 파일/폴더 경로를 객체처럼 편하게 다루기 위한 라이브러리입니다.
import wave               # wav 파일의 기본 정보를 확인하기 위한 파이썬 기본 라이브러리입니다.

# 여러 실행 환경을 고려해서 후보 경로를 여러 개 둡니다.
# 예를 들어 수업 자료 폴더에서는 ../data/S00007844.wav 일 수 있고,
# ChatGPT 실행 환경에서는 /mnt/data/S00007844.wav 일 수 있습니다.
audio_candidates = [
    Path('../data/S00007844.wav'),   # 노트북이 exercises 같은 하위 폴더에 있을 때 자주 쓰는 경로
    Path('data/S00007844.wav'),      # 현재 폴더 아래 data 폴더에 있을 때
    Path('./S00007844.wav'),         # 현재 폴더에 wav 파일이 바로 있을 때
    Path('/mnt/data/S00007844.wav'), # ChatGPT/샌드박스 환경에서 첨부 파일이 저장되는 경로
]

# 후보 경로 중에서 실제로 존재하는 첫 번째 경로를 AUDIO_PATH로 선택합니다.
# 만약 하나도 없으면 기본값으로 첫 번째 후보를 넣어둡니다.
# 이렇게 해두면 아래에서 FileNotFoundError를 통해 사용자에게 경로 수정 안내를 줄 수 있습니다.
AUDIO_PATH = next((p for p in audio_candidates if p.exists()), audio_candidates[0])

print('파일 존재 여부:', AUDIO_PATH.exists())
print('파일 경로:', AUDIO_PATH)

# 파일이 없으면 이후 모든 처리가 실패하므로 여기서 바로 중단합니다.
if not AUDIO_PATH.exists():
    raise FileNotFoundError(
        'S00007844.wav 파일을 찾지 못했습니다. AUDIO_PATH를 실제 wav 파일 경로로 수정하세요.'
    )

# wave.open()은 wav 파일의 헤더 정보를 읽을 때 사용합니다.
# 오디오 전체를 분석하는 것이 아니라, 파일의 기본 메타정보를 확인하는 단계입니다.
with wave.open(str(AUDIO_PATH), 'rb') as wf:
    channels = wf.getnchannels()       # 채널 수입니다. 1이면 모노, 2이면 스테레오입니다.
    sample_rate = wf.getframerate()    # 1초에 몇 개의 샘플이 있는지 나타냅니다. 예: 8000Hz, 16000Hz
    sample_width = wf.getsampwidth()   # 샘플 하나가 몇 byte인지 나타냅니다.
    frames = wf.getnframes()           # 전체 샘플 프레임 수입니다.
    duration = frames / sample_rate    # 전체 길이, 초 단위입니다.

print(f'채널 수: {channels}')
print(f'샘플링레이트: {sample_rate} Hz')
print(f'샘플 폭: {sample_width} bytes')
print(f'프레임 수: {frames}')
print(f'길이: {duration:.2f} 초')


파일 존재 여부: True
파일 경로: ..\data\S00007844.wav
채널 수: 1
샘플링레이트: 8000 Hz
샘플 폭: 2 bytes
길이: 267.33초 / 4.46분


## 2. 실행 장치 선택

faster-whisper는 CPU와 GPU 모두 사용할 수 있습니다.

- GPU 사용 가능: `device='cuda'`, `compute_type='float16'`
- CPU 사용: `device='cpu'`, `compute_type='int8'`

아래 코드는 GPU가 있으면 자동으로 GPU를 사용하고, 없으면 CPU를 사용합니다.


In [5]:
# ============================================
# 2. GPU / CPU 실행 장치 선택
# ============================================
# faster-whisper는 GPU가 있으면 훨씬 빠르게 실행됩니다.
# 하지만 GPU가 없어도 CPU로 실행할 수 있습니다.
#
# 이 셀은 nvidia-smi 명령이 있는지 확인해서 NVIDIA GPU 사용 가능 여부를 대략 판단합니다.
#
# device='cuda'  : NVIDIA GPU 사용
# device='cpu'   : CPU 사용
# compute_type='float16' : GPU에서 빠르고 효율적인 16비트 연산 사용
# compute_type='int8'    : CPU에서 메모리를 적게 쓰는 8비트 연산 사용

import shutil      # 특정 명령어가 시스템에 있는지 확인할 때 사용합니다.
import subprocess  # 외부 명령어를 실행할 때 사용합니다.


def has_nvidia_gpu() -> bool:
    """현재 환경에서 nvidia-smi 명령을 찾을 수 있으면 NVIDIA GPU가 있다고 판단합니다."""
    return shutil.which('nvidia-smi') is not None


# GPU 사용 가능 여부를 확인합니다.
USE_GPU = has_nvidia_gpu()

# GPU가 있으면 cuda/float16, 없으면 cpu/int8을 사용합니다.
if USE_GPU:
    device = 'cuda'
    compute_type = 'float16'
else:
    device = 'cpu'
    compute_type = 'int8'

print('사용 장치:', device)
print('compute_type:', compute_type)

# GPU가 있는 경우 nvidia-smi 결과를 일부 출력해서 어떤 GPU가 잡혔는지 확인합니다.
# 이 부분은 정보 확인용이므로 실패해도 노트북 전체 실행에는 큰 문제가 없습니다.
if USE_GPU:
    try:
        print(subprocess.check_output(['nvidia-smi'], text=True)[:1000])
    except Exception as e:
        print('nvidia-smi 확인 실패:', e)


사용 장치: cpu
compute_type: int8


## 3. Whisper 모델 선택

모델이 클수록 일반적으로 정확도는 좋아지지만 속도는 느려집니다.

| 모델 | 속도 | 정확도 | 추천 상황 |
|---|---:|---:|---|
| `tiny` | 매우 빠름 | 낮음 | 설치/동작 확인 |
| `base` | 빠름 | 보통 | 짧은 실습 |
| `small` | 적당함 | 좋음 | 일반 실습 추천 |
| `medium` | 느림 | 더 좋음 | 품질 우선 |
| `large-v3` | 가장 느림 | 가장 좋음 | GPU 환경 권장 |


In [11]:
# ============================================
# 3. faster-whisper 모델 로드
# ============================================
# 이 셀에서는 실제 음성 인식에 사용할 Whisper 모델을 불러옵니다.
#
# MODEL_SIZE는 모델 크기입니다.
# - tiny/base는 빠르지만 정확도가 낮을 수 있습니다.
# - small은 실습용으로 무난합니다.
# - medium/large-v3는 품질은 좋지만 느리고 GPU가 있으면 좋습니다.
#
# local_model_candidates는 로컬에 이미 다운로드된 모델이 있을 때 사용하기 위한 후보 경로입니다.
# 로컬 모델이 없으면 MODEL_SIZE 이름으로 Hugging Face에서 자동 다운로드합니다.

from faster_whisper import WhisperModel  # faster-whisper의 핵심 모델 클래스입니다.

# 처음 실습은 base 또는 small 추천입니다.
# 수업에서 빠른 실행을 원하면 'base', 품질을 조금 더 원하면 'small'을 추천합니다.
MODEL_SIZE = 'base'

# 로컬에 CTranslate2 형식의 faster-whisper 모델이 있을 수 있으므로 후보 경로를 여러 개 둡니다.
# 주의: 일반 openai-whisper 모델 폴더와 faster-whisper용 CTranslate2 모델 폴더는 구조가 다를 수 있습니다.
local_model_candidates = [
    Path('../models/whisper/base'),
    Path('../models/whisper_base'),
    Path('./models/whisper/base'),
    Path('./models/whisper_base'),
]

# 실제 존재하는 로컬 모델 경로를 찾습니다.
# 없으면 None이 됩니다.
local_model_path = next((p for p in local_model_candidates if p.exists()), None)

# 로컬 모델이 있으면 로컬 경로를 사용하고,
# 없으면 MODEL_SIZE 문자열을 사용해서 자동 다운로드 방식으로 로드합니다.
if local_model_path is not None:
    model_source = str(local_model_path)
    print('로컬 모델 사용:', model_source)
else:
    model_source = MODEL_SIZE
    print('로컬 모델이 없어 모델 이름으로 로드:', model_source)

# WhisperModel 객체를 생성합니다.
# 이 순간 모델 파일을 읽고, 필요하면 다운로드하고, GPU/CPU 메모리에 올립니다.
model = WhisperModel(
    model_source,
    device=device,
    compute_type=compute_type
)

print('모델 로드 완료:', model_source)


모델 로드 완료: base


## 4. 음성 → 텍스트 변환

아래 셀에서 실제 STT를 수행합니다.

주요 옵션:

- `language='ko'`: 한국어 음성으로 인식
- `beam_size=5`: 여러 후보를 비교해서 더 나은 결과 선택
- `vad_filter=True`: 음성이 아닌 구간을 어느 정도 제거
- `word_timestamps=True`: 단어 단위 시간 정보도 추출


In [12]:
# ============================================
# 4. 음성 파일을 텍스트로 변환하기
# ============================================
# model.transcribe()가 실제 STT를 수행합니다.
#
# 주요 옵션 설명:
# language='ko'
#   한국어 음성으로 인식하도록 지정합니다.
#
# beam_size=5
#   한 번에 하나의 후보만 보는 것이 아니라 여러 후보를 비교합니다.
#   숫자가 크면 품질이 좋아질 수 있지만 속도는 느려질 수 있습니다.
#
# vad_filter=True
#   VAD는 Voice Activity Detection, 즉 음성 구간 탐지입니다.
#   말이 없는 구간을 어느 정도 제외해서 인식 품질과 속도를 개선할 수 있습니다.
#
# word_timestamps=True
#   단어 단위 시작/끝 시간도 함께 추출합니다.
#   나중에 자막이나 화자분리 결합에 도움이 됩니다.

segments_iter, info = model.transcribe(
    str(AUDIO_PATH),
    language='ko',
    beam_size=5,
    vad_filter=True,
    word_timestamps=True
)

# info에는 감지 언어, 언어 확률, 전체 길이 같은 메타정보가 들어 있습니다.
print('감지 언어:', info.language)
print('언어 확률:', info.language_probability)
print('음성 길이:', info.duration, '초')

# segments_iter는 generator 형태입니다.
# 한 번에 list로 바꿔두면 이후 여러 셀에서 반복해서 사용할 수 있습니다.
segments = list(segments_iter)
print('인식된 세그먼트 수:', len(segments))


감지 언어: ko
언어 확률: 1
음성 길이: 267.33325 초
인식된 세그먼트 수: 75


## 5. 결과 확인

각 세그먼트는 다음 정보를 가집니다.

- 시작 시간
- 끝 시간
- 인식된 텍스트


In [15]:
# ============================================
# 5. 전사 결과 일부 확인
# ============================================
# Whisper는 긴 음성을 여러 segment로 나누어 인식합니다.
# 각 segment에는 start, end, text가 들어 있습니다.
#
# start : 이 문장이 시작된 시간, 초 단위
# end   : 이 문장이 끝난 시간, 초 단위
# text  : 인식된 문장

for i, seg in enumerate(segments[:20], start=1):
    # 앞에서 20개 세그먼트만 출력합니다.
    # 전체를 모두 출력하면 너무 길어질 수 있습니다.
    print(f'[{i:02d}] {seg.start:8.2f}s ~ {seg.end:8.2f}s | {seg.text.strip()}')


[01]     0.00s ~     1.42s | 상남사 김혜미입니다.
[02]     1.48s ~     2.30s | 무엇을 더 드릴까요?
[03]     3.74s ~     5.76s | 제가 학원 등록을 하고 싶은데
[04]     5.76s ~     8.12s | 학원에 간에 아는 정보도 없고
[05]     8.12s ~    11.30s | 인터넷에 찾아 봐도 정보가 부실한 것 같아서
[06]    11.56s ~    14.42s | 딱 CG 잡기 위해서 제가 연락을 드렸습니다.
[07]    15.76s ~    18.18s | 제가 이제 고등학교 3학년 되고
[08]    18.18s ~    22.30s | 주변 친구들도 이 광함은 학원에 다닌다고 해서
[09]    22.30s ~    25.00s | 교육 광함은 학원에 같이 등록하려고 해요.
[10]    26.34s ~    29.66s | 그런데 제가 지금까지 보험 모이고사 실력이
[11]    29.66s ~    35.32s | 구거는 이등급, 영원은 상등급, 수학은 이등급 정도 나와요.
[12]    35.32s ~    37.34s | 참고는 과학 참고를 보는데
[13]    37.34s ~    42.36s | 물리학원, 학원, 각각 이등급 씬 나옵니다.
[14]    42.96s ~    45.38s | 제 학 친구들은 특별한 반에 다니던데
[15]    45.38s ~    48.04s | 지금 성적으로는 가능할까요?
[16]    49.52s ~    50.36s | 아 그렇군요.
[17]    50.62s ~    52.24s | 사이트에 있는 조건표가 있던데
[18]    52.24s ~    54.86s | 그거 보니까 저 가능할 것 같은데, 맞아요?
[19]    56.02s ~    59.08s | 아 그리고 제가 다니는 친구들을
[20]    59.08s ~    63.58s | 말에 의하면 친구들은 레벨 테스트 보고 들어갔다고 해요.


## 6. 전체 텍스트 합치기

문장 구간별 결과를 하나의 전체 텍스트로 합칩니다.


In [17]:
# ============================================
# 6. 전체 텍스트 만들기
# ============================================
# segments는 여러 문장 조각으로 나뉘어 있습니다.
# 회의록 전체 문장을 보고 싶을 때는 각 segment의 text를 합치면 됩니다.

# seg.text.strip()은 앞뒤 공백을 제거합니다.
# ''.join(...)은 모든 문장을 하나의 문자열로 붙입니다.
full_text = ''.join(seg.text.strip() for seg in segments)

# 너무 길 수 있으므로 앞 2000자만 먼저 확인합니다.
print(full_text[:2000])
print('---')
print('전체 글자 수:', len(full_text))


상남사 김혜미입니다.무엇을 더 드릴까요?제가 학원 등록을 하고 싶은데학원에 간에 아는 정보도 없고인터넷에 찾아 봐도 정보가 부실한 것 같아서딱 CG 잡기 위해서 제가 연락을 드렸습니다.제가 이제 고등학교 3학년 되고주변 친구들도 이 광함은 학원에 다닌다고 해서교육 광함은 학원에 같이 등록하려고 해요.그런데 제가 지금까지 보험 모이고사 실력이구거는 이등급, 영원은 상등급, 수학은 이등급 정도 나와요.참고는 과학 참고를 보는데물리학원, 학원, 각각 이등급 씬 나옵니다.제 학 친구들은 특별한 반에 다니던데지금 성적으로는 가능할까요?아 그렇군요.사이트에 있는 조건표가 있던데그거 보니까 저 가능할 것 같은데, 맞아요?아 그리고 제가 다니는 친구들을말에 의하면 친구들은 레벨 테스트 보고 들어갔다고 해요.지금은 코로나19 상황으로 인해서 거의 밀집 금지하던데지금도 현재 레벨 테스트를 진행하나요?아니요 뭐 읽어서 성적만 봅니다.뭐 읽어서 성적만 보는군요.사이트에는 사진 버전이 옛날 버전이라지금도 바뀌었는지 궁금했어요.사이트에 신청 방법을 보니까제가 직접 가도 되고 전화로 해도 된다고 하던데요.맞아요?네.동록하러 가고 싶은데제가 갈 경우에는 홈페이지에서 방문 가능 시간을 확인했거든요.평일 오전 9시부터 6시까지로 봤는데방문도 동일한 시간으로 생각해도 될까요?아 그래요?지금 코로나19 상황으로 시간 변동 상황은 없겠죠?네 없어요.제가 듣고 싶은 과목이 영어랑 과학 당구인데이런 과목이 있는 반이 특별한 반인데 맞을까요?네 맞아요.그런데 구거도 들어야겠는데특별한 반에는 과목은 없네요.이걸 듣고 싶으면 일반 반에 들어가야 하는데 맞나요?그렇다면 특별한 반이 듣는 시간이 일반 반을 각 과목을 시간보다5시간 더 많다고 하는데그게 적혀있는 표 아랫부분에 추구 변동 될 수 있음을 제가 봤어요.지금 변동 되었나요?안녕?알겠습니다.그렇다면 수강류 차이가 일반 반이 125만 4,990번특별 반은 150만 8,990번이라 해요.지금도 변동 상황은 없겠죠?그리고 모일고사 3,5등급 8에 따라 장악금

## 7. 결과를 CSV로 저장

시간 정보와 텍스트를 표 형태로 저장합니다.

나중에 화자분리 결과와 결합할 때는 이 CSV의 `start`, `end` 시간이 중요합니다.


In [18]:
# ============================================
# 7-1. Whisper 전사 결과를 표 형태로 만들기
# ============================================
# 나중에 화자분리 결과와 합치려면 텍스트뿐 아니라 시간이 중요합니다.
# 그래서 각 segment를 아래 형태의 표로 만듭니다.
#
# segment_id | start | end | duration | text

import pandas as pd  # 표 형태 데이터를 만들기 위해 사용합니다.

rows = []

# enumerate(..., start=1)은 1부터 번호를 붙이기 위한 코드입니다.
for idx, seg in enumerate(segments, start=1):
    rows.append({
        'segment_id': idx,              # Whisper가 나눈 문장 번호
        'start': seg.start,             # 문장 시작 시간
        'end': seg.end,                 # 문장 종료 시간
        'duration': seg.end - seg.start,# 문장 길이
        'text': seg.text.strip()        # 인식된 텍스트
    })

# rows 리스트를 pandas DataFrame으로 변환합니다.
transcript_df = pd.DataFrame(rows)

# 앞 10개 행을 확인합니다.
transcript_df.head(10)


,segment_id,start,end,duration,text
0,1,0.00,1.42,1.42,상남사 김혜미입니다.
1,2,1.48,2.30,0.82,무엇을 더 드릴까요?
2,3,3.74,5.76,2.02,제가 학원 등록을 하고 싶은데
3,4,5.76,8.12,2.36,학원에 간에 아는 정보도 없고
4,5,8.12,11.30,3.18,인터넷에 찾아 봐도 정보가 부실한 것 같아서
5,6,11.56,14.42,2.86,딱 CG 잡기 위해서 제가 연락을 드렸습니다.
6,7,15.76,18.18,2.42,제가 이제 고등학교 3학년 되고
7,8,18.18,22.30,4.12,주변 친구들도 이 광함은 학원에 다닌다고 해서
8,9,22.30,25.00,2.70,교육 광함은 학원에 같이 등록하려고 해요.
9,10,26.34,29.66,3.32,그런데 제가 지금까지 보험 모이고사 실력이


In [30]:
# ============================================
# 7-2. Whisper 전사 결과 저장
# ============================================
# 결과를 파일로 저장해두면 다음 실습이나 엑셀 확인에 사용할 수 있습니다.
#
# CSV : 표 형태 저장
# TXT : 전체 텍스트 저장
# SRT : 자막 파일 저장, 아래 셀에서 사용

OUTPUT_DIR = Path('../output/faster_whisper_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)  # 폴더가 없으면 자동으로 만듭니다.

csv_path = OUTPUT_DIR / 'S00007844_faster_whisper_transcript.csv'
txt_path = OUTPUT_DIR / 'S00007844_faster_whisper_transcript.txt'
srt_path = OUTPUT_DIR / 'S00007844_faster_whisper_transcript.srt'

# encoding='utf-8-sig'는 엑셀에서 한글 CSV를 열 때 깨짐을 줄이기 위해 사용합니다.
transcript_df.to_csv(csv_path, index=False, encoding='utf-8-sig')

# 전체 텍스트는 txt 파일로 저장합니다.
txt_path.write_text(full_text, encoding='utf-8')

print('CSV 저장:', csv_path)
print('TXT 저장:', txt_path)


CSV 저장: ..\output\faster_whisper_output\S00007844_faster_whisper_transcript.csv
TXT 저장: ..\output\faster_whisper_output\S00007844_faster_whisper_transcript.txt


## 8. SRT 자막 파일로 저장

영상/음성 플레이어에서 자막처럼 볼 수 있는 `.srt` 파일을 만듭니다.


In [31]:
# ============================================
# 8. SRT 자막 파일 만들기
# ============================================
# SRT 파일은 아래와 같은 형태의 자막 파일입니다.
#
# 1
# 00:00:00,000 --> 00:00:03,200
# 안녕하세요
#
# SRT에서는 시간을 HH:MM:SS,mmm 형식으로 써야 하므로
# seconds_to_srt_time() 함수로 초 단위 시간을 변환합니다.


def seconds_to_srt_time(seconds: float) -> str:
    """초 단위 시간을 SRT 자막 시간 형식으로 바꿉니다."""
    hours = int(seconds // 3600)              # 전체 초에서 시간 부분 계산
    minutes = int((seconds % 3600) // 60)     # 남은 초에서 분 부분 계산
    secs = int(seconds % 60)                  # 초 부분 계산
    millis = int(round((seconds - int(seconds)) * 1000))  # 밀리초 계산
    return f'{hours:02d}:{minutes:02d}:{secs:02d},{millis:03d}'


srt_lines = []

# 각 Whisper segment를 SRT 자막 한 블록으로 변환합니다.
for idx, seg in enumerate(segments, start=1):
    srt_lines.append(str(idx))
    srt_lines.append(f'{seconds_to_srt_time(seg.start)} --> {seconds_to_srt_time(seg.end)}')
    srt_lines.append(seg.text.strip())
    srt_lines.append('')  # SRT 블록 사이에는 빈 줄이 필요합니다.

# 줄들을 \n으로 합쳐서 파일에 저장합니다.
srt_path.write_text('\n'.join(srt_lines), encoding='utf-8')
print('SRT 저장:', srt_path)


SRT 저장: ..\output\faster_whisper_output\S00007844_faster_whisper_transcript.srt


## 9. 단어 단위 타임스탬프 저장

`word_timestamps=True` 옵션을 켰기 때문에 단어별 시간 정보도 저장할 수 있습니다.

한국어는 띄어쓰기/발음 특성 때문에 단어 단위 시간이 완벽하지 않을 수 있지만, 자막 정렬이나 화자분리 결합에는 도움이 됩니다.


In [32]:
# ============================================
# 9. 단어 단위 타임스탬프 저장
# ============================================
# model.transcribe(..., word_timestamps=True)를 사용했기 때문에
# segment 안에 단어별 시작/종료 시간 정보가 들어올 수 있습니다.
#
# 단, 한국어는 영어처럼 단어 경계가 항상 명확하지 않아서
# 단어 단위 시간이 완벽하지 않을 수 있습니다.

word_rows = []

# 세그먼트별로 단어 정보를 꺼냅니다.
for seg_idx, seg in enumerate(segments, start=1):
    # 환경이나 모델 옵션에 따라 seg.words가 없을 수 있으므로 확인합니다.
    if seg.words is None:
        continue

    for word_idx, word in enumerate(seg.words, start=1):
        word_rows.append({
            'segment_id': seg_idx,          # 어떤 문장 세그먼트에 속한 단어인지
            'word_id': word_idx,            # 해당 세그먼트 안에서 몇 번째 단어인지
            'start': word.start,            # 단어 시작 시간
            'end': word.end,                # 단어 종료 시간
            'word': word.word.strip(),      # 인식된 단어
            'probability': word.probability # 모델이 추정한 단어 신뢰도
        })

word_df = pd.DataFrame(word_rows)
word_csv_path = OUTPUT_DIR / 'S00007844_faster_whisper_words.csv'
word_df.to_csv(word_csv_path, index=False, encoding='utf-8-sig')

print('단어 단위 CSV 저장:', word_csv_path)
word_df.head(20)


단어 단위 CSV 저장: ..\output\faster_whisper_output\S00007844_faster_whisper_words.csv


,segment_id,word_id,start,end,word,probability
0,1,1,0.00,0.62,상남사,0.496823
1,1,2,0.62,1.42,김혜미입니다.,0.482084
2,2,1,1.48,1.82,무엇을,0.616473
3,2,2,1.82,1.90,더,0.407750
4,2,3,1.90,2.30,드릴까요?,0.963769
5,3,1,3.74,4.12,제가,0.684066
6,3,2,4.12,4.62,학원,0.911584
7,3,3,4.62,5.06,등록을,0.773316
8,3,4,5.06,5.22,하고,0.994599
9,3,5,5.22,5.76,싶은데,0.951298


## 10. MFCC + KMeans로 화자분리 표 만들기

Whisper는 텍스트를 만들지만, 기본적으로 `누가 말했는지`를 안정적으로 알려주지는 않습니다.

그래서 같은 음성 파일에서 MFCC 특징을 뽑고, KMeans로 비슷한 목소리 구간끼리 묶어서 화자분리 표를 만듭니다.

아래 결과는 다음 형태의 표가 됩니다.

```text
start | end | speaker | rms
0.00  | 1.50 | Speaker_00 | ...
0.75  | 2.25 | Speaker_00 | ...
2.25  | 3.75 | Speaker_01 | ...
```

In [ ]:
# ============================================
# 10. MFCC + KMeans 화자분리 설정
# ============================================
# 여기부터는 Whisper가 만든 텍스트가 아니라,
# 같은 음성 파일에서 목소리 특징을 뽑아 화자를 추정하는 부분입니다.
#
# 중요한 개념:
# - MFCC는 음성의 주파수 특성을 숫자로 요약한 특징입니다.
# - KMeans는 비슷한 숫자 특징끼리 묶는 군집화 알고리즘입니다.
# - 따라서 MFCC + KMeans는 "비슷한 목소리 구간끼리 묶는 실습용 화자분리"입니다.

import numpy as np
import pandas as pd
import librosa

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 예상 화자 수입니다.
# 실제 음성에 사람이 2명이면 2, 3명이면 3으로 바꿔서 다시 실행하세요.
# KMeans는 이 숫자만큼 군집을 만들기 때문에 매우 중요한 설정입니다.
N_SPEAKERS = 2

# MFCC를 뽑을 음성 구간 길이입니다.
# 1.5초 단위로 목소리 특징을 추출한다는 뜻입니다.
# 너무 짧으면 특징이 불안정하고, 너무 길면 화자 전환 지점이 뭉개질 수 있습니다.
CHUNK_SECONDS = 1.5

# 다음 구간으로 이동하는 간격입니다.
# 0.75초면 1.5초 창을 절반씩 겹치면서 분석합니다.
# 겹쳐서 보면 더 촘촘하게 화자 변화를 볼 수 있지만, 계산량은 늘어납니다.
HOP_SECONDS = 0.75

# 너무 조용한 구간은 말소리가 아닐 가능성이 높으므로 제외합니다.
# 녹음 음량에 따라 0.003 ~ 0.01 사이로 조정해 볼 수 있습니다.
RMS_THRESHOLD = 0.005

# MFCC 계수 개수입니다.
# 일반적으로 13 또는 20을 많이 씁니다.
# 여기서는 조금 더 많은 정보를 쓰기 위해 20을 사용합니다.
N_MFCC = 20

# 앞에서 만든 faster-whisper 출력 폴더를 그대로 사용합니다.
OUTPUT_DIR = Path('../output/faster_whisper_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('예상 화자 수:', N_SPEAKERS)
print('청크 길이:', CHUNK_SECONDS, '초')
print('이동 간격:', HOP_SECONDS, '초')
print('출력 폴더:', OUTPUT_DIR)


### 10-1. 오디오를 읽고 MFCC 특징 추출

각 구간마다 아래 특징을 뽑습니다.

- MFCC 평균
- MFCC 표준편차
- Delta MFCC 평균
- Delta MFCC 표준편차
- RMS 에너지

KMeans는 텍스트를 보는 것이 아니라, 이 숫자 특징을 보고 비슷한 목소리끼리 묶습니다.

In [ ]:
# ============================================
# 10-1. MFCC/KMeans에 사용할 특징 추출 함수
# ============================================
# 이 함수는 오디오 파일을 작은 구간으로 자른 다음,
# 각 구간에서 화자 구분에 사용할 숫자 특징을 뽑습니다.
#
# 반환되는 DataFrame 예:
# start | end | duration | rms | feat_00 | feat_01 | ...
#
# 여기서 feat_00, feat_01 같은 값들이 KMeans 입력값이 됩니다.


def extract_mfcc_kmeans_features(
    audio_path: Path,
    sr_target: int = 16000,
    chunk_seconds: float = 1.5,
    hop_seconds: float = 0.75,
    rms_threshold: float = 0.005,
    n_mfcc: int = 20,
) -> pd.DataFrame:
    """
    오디오를 작은 구간으로 자르고, 각 구간에서 MFCC 기반 특징을 추출합니다.

    audio_path:
        분석할 원본 wav 파일 경로입니다.

    sr_target:
        오디오를 몇 Hz로 읽을지 정합니다.
        음성인식/음성분석 실습에서는 16000Hz를 자주 사용합니다.

    chunk_seconds:
        한 번에 분석할 구간 길이입니다.
        예: 1.5면 1.5초 단위로 목소리 특징을 봅니다.

    hop_seconds:
        다음 구간으로 이동하는 간격입니다.
        예: 0.75면 0.75초씩 이동하므로 구간이 서로 겹칩니다.

    rms_threshold:
        너무 조용한 구간을 제외하기 위한 임계값입니다.

    n_mfcc:
        추출할 MFCC 계수 개수입니다.
    """

    # librosa.load()로 오디오를 읽습니다.
    # mono=True는 스테레오를 모노로 합친다는 뜻입니다.
    # sr=sr_target은 원본 샘플링레이트와 관계없이 16000Hz로 맞춘다는 뜻입니다.
    y, sr = librosa.load(audio_path, sr=sr_target, mono=True)

    # 초 단위 길이를 샘플 개수로 바꿉니다.
    # 예: sr=16000, chunk_seconds=1.5라면 24000 샘플입니다.
    chunk_samples = int(chunk_seconds * sr)
    hop_samples = int(hop_seconds * sr)

    # 각 구간의 특징 행을 저장할 리스트입니다.
    rows = []

    # start_sample을 hop_samples만큼 이동시키면서 전체 오디오를 훑습니다.
    for start_sample in range(0, max(1, len(y) - chunk_samples + 1), hop_samples):
        end_sample = min(start_sample + chunk_samples, len(y))
        chunk = y[start_sample:end_sample]

        # 마지막 조각이 너무 짧으면 특징이 불안정하므로 건너뜁니다.
        if len(chunk) < int(0.4 * sr):
            continue

        # 샘플 위치를 초 단위 시간으로 변환합니다.
        start_sec = start_sample / sr
        end_sec = end_sample / sr

        # RMS는 Root Mean Square의 약자로, 구간의 평균 음량에 가깝게 볼 수 있습니다.
        # 값이 너무 작으면 말소리가 아니라 무음/잡음일 가능성이 큽니다.
        rms = float(librosa.feature.rms(y=chunk)[0].mean())

        # 너무 조용한 구간은 KMeans에 넣지 않습니다.
        # 무음까지 군집화하면 Speaker_00/01이 무음 차이로 나뉠 수 있습니다.
        if rms < rms_threshold:
            continue

        # MFCC는 목소리의 주파수 특성을 압축해서 표현한 숫자 특징입니다.
        # 화자마다 성도, 발음, 음색이 다르기 때문에 MFCC가 달라질 수 있습니다.
        mfcc = librosa.feature.mfcc(y=chunk, sr=sr, n_mfcc=n_mfcc)

        # Delta MFCC는 MFCC가 시간에 따라 어떻게 변하는지를 나타냅니다.
        # 단순 음색뿐 아니라 발음 변화 패턴을 조금 더 반영할 수 있습니다.
        delta = librosa.feature.delta(mfcc)

        # KMeans는 고정 길이 숫자 벡터를 입력으로 받습니다.
        # 그래서 시간축으로 펼쳐져 있는 MFCC를 평균/표준편차로 요약합니다.
        #
        # 사용 특징:
        # 1) MFCC 평균
        # 2) MFCC 표준편차
        # 3) Delta MFCC 평균
        # 4) Delta MFCC 표준편차
        # 5) RMS 음량
        feature = np.concatenate([
            mfcc.mean(axis=1),
            mfcc.std(axis=1),
            delta.mean(axis=1),
            delta.std(axis=1),
            np.array([rms]),
        ])

        # 기본 시간 정보와 음량 정보를 먼저 저장합니다.
        row = {
            'start': start_sec,
            'end': end_sec,
            'duration': end_sec - start_sec,
            'rms': rms,
        }

        # feature 벡터의 각 숫자를 feat_00, feat_01 ... 형태의 컬럼으로 저장합니다.
        for i, value in enumerate(feature):
            row[f'feat_{i:02d}'] = float(value)

        rows.append(row)

    # 리스트를 DataFrame으로 바꿔서 반환합니다.
    feature_df = pd.DataFrame(rows)
    return feature_df


# 실제로 현재 AUDIO_PATH에 대해 MFCC 특징을 추출합니다.
mfcc_feature_df = extract_mfcc_kmeans_features(
    AUDIO_PATH,
    chunk_seconds=CHUNK_SECONDS,
    hop_seconds=HOP_SECONDS,
    rms_threshold=RMS_THRESHOLD,
    n_mfcc=N_MFCC,
)

print('MFCC 특징 구간 수:', len(mfcc_feature_df))
mfcc_feature_df.head()


### 10-2. KMeans로 비슷한 목소리끼리 묶기

이 단계에서 `Speaker_00`, `Speaker_01` 같은 라벨이 만들어집니다.

주의할 점은 KMeans의 Speaker 번호는 실제 사람 이름이 아닙니다.  
`Speaker_00 = 김철수` 같은 의미가 아니라, 단지 비슷한 목소리로 묶인 군집 번호입니다.

In [ ]:
# ============================================
# 10-2. KMeans로 화자 군집화하기
# ============================================
# 이 셀에서는 앞에서 뽑은 MFCC 특징을 KMeans에 넣습니다.
#
# KMeans가 하는 일:
# - feat_00, feat_01 ... 같은 숫자 특징을 보고
# - 비슷한 특징끼리 N_SPEAKERS개 그룹으로 묶습니다.
# - 그 그룹을 Speaker_00, Speaker_01처럼 이름 붙입니다.
#
# 주의:
# Speaker_00, Speaker_01은 실제 사람 이름이 아닙니다.
# 단지 비슷한 목소리 특징으로 묶인 군집 번호입니다.

# 특징이 하나도 없으면 이후 KMeans를 실행할 수 없으므로 에러를 냅니다.
if mfcc_feature_df.empty:
    raise ValueError('MFCC 특징이 없습니다. RMS_THRESHOLD를 낮추거나 AUDIO_PATH를 확인하세요.')

# KMeans에 넣을 feature 컬럼만 고릅니다.
# start/end/rms 같은 메타정보는 군집화 입력에서 제외합니다.
feature_cols = [c for c in mfcc_feature_df.columns if c.startswith('feat_')]
X = mfcc_feature_df[feature_cols].values

# 특징마다 값의 범위가 다를 수 있습니다.
# 예를 들어 어떤 특징은 -500~500, 어떤 특징은 0~1일 수 있습니다.
# KMeans는 거리 기반 알고리즘이라 값 범위가 큰 특징에 더 끌릴 수 있습니다.
# 그래서 StandardScaler로 평균 0, 표준편차 1 형태로 맞춥니다.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 데이터 구간 수보다 화자 수가 클 수는 없으므로 안전하게 보정합니다.
# 예: 구간이 1개인데 화자 수를 2로 두면 KMeans가 실패합니다.
n_clusters = min(N_SPEAKERS, len(mfcc_feature_df))

# KMeans 모델을 만듭니다.
# random_state=42는 실행할 때마다 결과가 너무 달라지지 않게 고정하는 값입니다.
# n_init='auto'는 sklearn 최신 버전에서 권장되는 초기화 반복 설정입니다.
kmeans = KMeans(
    n_clusters=n_clusters,
    random_state=42,
    n_init='auto',
)

# 각 구간이 어떤 군집에 속하는지 예측합니다.
# labels 예: [0, 0, 1, 1, 0, ...]
labels = kmeans.fit_predict(X_scaled)

# 숫자 라벨을 보기 좋은 화자 라벨로 바꿉니다.
# 0 -> Speaker_00, 1 -> Speaker_01
mfcc_feature_df['speaker'] = [f'Speaker_{label:02d}' for label in labels]

# 화자분리 표에 필요한 컬럼만 추립니다.
diar_df = mfcc_feature_df[['start', 'end', 'duration', 'speaker', 'rms']].copy()

# 원본 화자분리 표를 CSV로 저장합니다.
# 이 표는 CHUNK_SECONDS/HOP_SECONDS 기준으로 잘게 나뉜 표입니다.
diarization_csv_path = OUTPUT_DIR / 'S00007844_mfcc_kmeans_segments.csv'
diar_df.to_csv(diarization_csv_path, index=False, encoding='utf-8-sig')

print('MFCC/KMeans 원본 화자분리 표 저장:', diarization_csv_path)
diar_df.head(20)


### 10-3. 같은 화자가 이어지는 구간 병합

위 표는 1.5초 단위로 쪼개져 있어서 회의록으로 보기에는 너무 잘게 나뉩니다.

그래서 같은 화자가 연속으로 나오면 하나의 구간으로 합칩니다.

In [ ]:
# ============================================
# 10-3. 같은 화자가 이어지는 구간 병합
# ============================================
# KMeans 결과는 1.5초 단위처럼 매우 잘게 나뉩니다.
# 회의록으로 보기에는 너무 촘촘하므로,
# 같은 화자가 연속해서 나오면 하나의 큰 구간으로 합칩니다.
#
# 예:
# 0.00~1.50 Speaker_00
# 0.75~2.25 Speaker_00
# 1.50~3.00 Speaker_00
#
# 위와 같은 구간은 하나의 Speaker_00 구간으로 합칠 수 있습니다.


def merge_same_speaker_segments(diar_df: pd.DataFrame, gap_threshold: float = 0.6) -> pd.DataFrame:
    """
    같은 화자가 연속으로 나온 구간을 하나로 병합합니다.

    diar_df:
        start, end, speaker 컬럼을 가진 화자분리 표입니다.

    gap_threshold:
        두 구간 사이의 빈 시간이 이 값 이하이면 이어진 것으로 봅니다.
        예: 0.6이면 두 구간 사이가 0.6초 이하로 벌어져 있어도 병합합니다.
    """

    # 입력 표가 비어 있으면 그대로 반환합니다.
    if diar_df.empty:
        return diar_df.copy()

    # 시간 순서대로 정렬합니다.
    # 병합은 앞에서 뒤로 순서대로 비교해야 하기 때문입니다.
    diar_df = diar_df.sort_values('start').reset_index(drop=True)

    # 병합된 결과를 담을 리스트입니다.
    merged = []

    for _, row in diar_df.iterrows():
        start = float(row['start'])
        end = float(row['end'])
        speaker = str(row['speaker'])
        rms = float(row.get('rms', 0.0))

        # 첫 번째 구간은 비교할 이전 구간이 없으므로 바로 추가합니다.
        if not merged:
            merged.append({
                'start': start,
                'end': end,
                'speaker': speaker,
                'rms_mean': rms,
                'segment_count': 1,
            })
            continue

        # 직전에 병합된 마지막 구간을 가져옵니다.
        last = merged[-1]

        # 병합 조건:
        # 1) 현재 구간의 speaker가 직전 구간 speaker와 같고
        # 2) 현재 start와 직전 end 사이 간격이 gap_threshold 이하이면
        # 같은 발화 흐름으로 보고 병합합니다.
        if last['speaker'] == speaker and start - last['end'] <= gap_threshold:
            count = last['segment_count']

            # 종료 시간은 더 뒤쪽 시간으로 확장합니다.
            last['end'] = max(last['end'], end)

            # rms_mean은 기존 평균과 현재 rms를 누적 평균 방식으로 갱신합니다.
            last['rms_mean'] = (last['rms_mean'] * count + rms) / (count + 1)

            # 몇 개의 작은 구간이 합쳐졌는지 기록합니다.
            last['segment_count'] = count + 1
        else:
            # 화자가 다르거나 간격이 너무 벌어져 있으면 새 구간으로 추가합니다.
            merged.append({
                'start': start,
                'end': end,
                'speaker': speaker,
                'rms_mean': rms,
                'segment_count': 1,
            })

    # 병합 결과를 DataFrame으로 바꿉니다.
    merged_df = pd.DataFrame(merged)

    # 병합된 구간 길이를 계산합니다.
    merged_df['duration'] = merged_df['end'] - merged_df['start']

    # 보기 좋은 컬럼 순서로 반환합니다.
    return merged_df[['start', 'end', 'duration', 'speaker', 'rms_mean', 'segment_count']]


# 실제 병합 수행
merged_diar_df = merge_same_speaker_segments(diar_df, gap_threshold=0.6)

# 병합된 화자분리 표 저장
merged_diarization_csv_path = OUTPUT_DIR / 'S00007844_mfcc_kmeans_segments_merged.csv'
merged_diar_df.to_csv(merged_diarization_csv_path, index=False, encoding='utf-8-sig')

print('병합된 화자분리 표 저장:', merged_diarization_csv_path)
merged_diar_df.head(30)


### 10-4. 화자별 발화 시간 요약표

화자분리 결과가 대략 어떻게 나왔는지 확인하기 위한 요약표입니다.

In [ ]:
# ============================================
# 10-4. 화자별 발화 시간 요약표 만들기
# ============================================
# merged_diar_df에는 병합된 화자 구간이 들어 있습니다.
# 이 표를 groupby해서 화자별 전체 발화 시간, 구간 개수, 평균 구간 길이를 계산합니다.
#
# 이 요약표는 결과가 대략 말이 되는지 확인하는 용도입니다.
# 예를 들어 Speaker_00만 너무 많고 Speaker_01이 거의 없다면
# N_SPEAKERS나 RMS_THRESHOLD, CHUNK_SECONDS를 조정해 볼 수 있습니다.

speaker_summary_df = (
    merged_diar_df
    .groupby('speaker', as_index=False)
    .agg(
        total_speech_sec=('duration', 'sum'), # 화자별 총 발화 시간, 초 단위
        segment_count=('speaker', 'count'),   # 화자별 병합 구간 개수
        avg_segment_sec=('duration', 'mean'), # 화자별 평균 구간 길이
    )
    .sort_values('speaker')
)

# 분 단위 시간도 추가합니다.
speaker_summary_df['total_speech_min'] = speaker_summary_df['total_speech_sec'] / 60

# CSV로 저장합니다.
speaker_summary_csv_path = OUTPUT_DIR / 'S00007844_mfcc_kmeans_speaker_summary.csv'
speaker_summary_df.to_csv(speaker_summary_csv_path, index=False, encoding='utf-8-sig')

print('화자별 요약표 저장:', speaker_summary_csv_path)
speaker_summary_df


## 11. Whisper 전사 결과와 MFCC/KMeans 화자분리 결과 결합

이제 두 표를 시간 기준으로 합칩니다.

- `transcript_df`: Whisper가 만든 텍스트 표
- `merged_diar_df`: MFCC/KMeans가 만든 화자분리 표

Whisper 문장 구간과 가장 많이 겹치는 화자를 찾아서 `speaker` 컬럼을 붙입니다.

In [ ]:
# ============================================
# 11. Whisper 텍스트와 화자분리 표 결합
# ============================================
# 지금까지 만든 표는 두 개입니다.
#
# 1) transcript_df
#    Whisper가 만든 텍스트 표입니다.
#    컬럼 예: segment_id, start, end, text
#
# 2) merged_diar_df
#    MFCC/KMeans가 만든 화자분리 표입니다.
#    컬럼 예: start, end, speaker
#
# 두 표 모두 start/end 시간이 있으므로,
# Whisper 문장 시간과 가장 많이 겹치는 화자를 찾아 붙입니다.

from typing import Optional


def overlap_duration(a_start: float, a_end: float, b_start: float, b_end: float) -> float:
    """
    두 시간 구간이 겹치는 길이를 초 단위로 계산합니다.

    예:
    A 구간: 0초 ~ 5초
    B 구간: 3초 ~ 7초
    겹치는 구간은 3초 ~ 5초이므로 overlap은 2초입니다.
    """

    # min(a_end, b_end)는 두 구간 끝점 중 더 이른 끝점입니다.
    # max(a_start, b_start)는 두 구간 시작점 중 더 늦은 시작점입니다.
    # 이 둘의 차이가 양수면 겹친 것이고, 음수면 겹치지 않은 것입니다.
    return max(0.0, min(a_end, b_end) - max(a_start, b_start))


def attach_speaker_by_overlap(transcript_df: pd.DataFrame, diar_df: pd.DataFrame) -> pd.DataFrame:
    """
    STT 문장 구간마다 가장 많이 겹치는 화자 라벨을 붙입니다.

    처리 방식:
    1. Whisper 문장 하나를 선택합니다.
    2. 모든 화자분리 구간과 시간 겹침을 계산합니다.
    3. 가장 많이 겹친 화자를 해당 문장의 speaker로 붙입니다.
    """

    results = []

    # Whisper 문장 하나씩 반복합니다.
    for _, t in transcript_df.iterrows():
        best_speaker: Optional[str] = None
        best_overlap = 0.0

        # 현재 Whisper 문장의 시작/종료 시간입니다.
        t_start = float(t['start'])
        t_end = float(t['end'])

        # 현재 Whisper 문장과 모든 화자분리 구간을 비교합니다.
        for _, d in diar_df.iterrows():
            ov = overlap_duration(
                t_start,
                t_end,
                float(d['start']),
                float(d['end']),
            )

            # 지금까지 본 화자 구간 중 가장 많이 겹친 화자를 저장합니다.
            if ov > best_overlap:
                best_overlap = ov
                best_speaker = str(d['speaker'])

        # Whisper 원본 row를 dict로 복사한 뒤 speaker 정보를 추가합니다.
        row = t.to_dict()
        row['speaker'] = best_speaker if best_speaker is not None else 'Unknown'
        row['speaker_overlap_sec'] = best_overlap
        results.append(row)

    return pd.DataFrame(results)


# 실제로 Whisper 전사 표와 병합된 화자분리 표를 결합합니다.
speaker_transcript_df = attach_speaker_by_overlap(transcript_df, merged_diar_df)

# 보기 좋은 컬럼 순서로 정리합니다.
front_cols = ['segment_id', 'start', 'end', 'duration', 'speaker', 'text', 'speaker_overlap_sec']
other_cols = [c for c in speaker_transcript_df.columns if c not in front_cols]
speaker_transcript_df = speaker_transcript_df[front_cols + other_cols]

# 최종 결과 CSV 저장
speaker_transcript_csv_path = OUTPUT_DIR / 'S00007844_speaker_transcript.csv'
speaker_transcript_df.to_csv(speaker_transcript_csv_path, index=False, encoding='utf-8-sig')

print('화자 포함 전사 결과 저장:', speaker_transcript_csv_path)
speaker_transcript_df.head(30)


## 12. 최종 회의록 형태로 출력

마지막으로 `[Speaker_00] 발화 내용` 형태로 출력합니다.

In [ ]:
# ============================================
# 12. 최종 회의록 형태로 화면 출력
# ============================================
# 최종 표 speaker_transcript_df를 사람이 보기 쉬운 형태로 출력합니다.
#
# 출력 예:
# [   0.00 ~    3.20] Speaker_00: 안녕하세요 회의를 시작하겠습니다
# [   3.20 ~    5.80] Speaker_01: 네 자료 공유하겠습니다

for _, row in speaker_transcript_df.iterrows():
    start = float(row['start'])
    end = float(row['end'])
    speaker = row['speaker']
    text = str(row['text']).strip()

    # 텍스트가 비어 있는 행은 출력하지 않습니다.
    if not text:
        continue

    print(f'[{start:7.2f} ~ {end:7.2f}] {speaker}: {text}')


## 13. 수업용 정리

이 노트북의 전체 흐름은 다음과 같습니다.

```text
S00007844.wav
   ├─ faster-whisper → transcript_df → 무슨 말을 했는지
   └─ MFCC + KMeans → merged_diar_df → 누가 말했는지 추정
                  ↓
        시간 겹침 기준으로 결합
                  ↓
        S00007844_speaker_transcript.csv
```

중요한 점은 다음입니다.

- Whisper는 텍스트 변환용입니다.
- MFCC는 목소리 특징 추출용입니다.
- KMeans는 비슷한 목소리 구간을 묶는 용도입니다.
- 최종 화자별 회의록은 Whisper 결과와 MFCC/KMeans 결과를 시간 기준으로 합쳐서 만듭니다.

MFCC + KMeans 방식은 간단한 실습용입니다. 실제 회의록 서비스 품질을 높이려면 `pyannote.audio`, `WhisperX`, `NVIDIA NeMo diarization`을 검토하는 것이 좋습니다.